In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

# 1. BUSINESS UNDERSTANDING
## **Tujuan bisnis:**
Tim marketing ingin memahami kelompok-kelompok pelanggan yang ada berdasarkan usia, pendapatan tahunan, dan skor belanja, agar bisa merancang strategi promosi/produk yang lebih tepat sasaran untuk tiap kelompok (bukan strategi "satu untuk semua pelanggan").

## **Pertanyaan bisnis yang ingin dijawab:**
* Ada berapa kelompok (segmen) pelanggan yang berbeda karakteristik?
* Apa ciri khas tiap segmen (usia, pendapatan, pola belanja)?
* Strategi marketing apa yang cocok untuk tiap segmen?

## **Tujuan data mining (data mining goal):**
Mengelompokkan (clustering) pelanggan ke dalam beberapa segmen menggunakan algoritma K-Means, berdasarkan atribut numerik yang tersedia (Usia, Pendapatan_Tahunan_Juta, Skor_Belanja).

# 2. DATA UNDERSTANDING
Mengumpulkan data, mendeskripsikan data, dan mengeksplorasi data untuk mengenali kualitas data (missing value, duplikasi, anomali) sebelum diproses lebih lanjut.

Mengumpulkan data (Collect Initial Data). Ambil data dari .csv

In [6]:
# Dataset baru menggunakan pemisah titik koma (;)
df_raw = pd.read_csv('Data_Rumah_Tangga_Rapi.csv', sep=';')
print(df_raw.info())
df_raw.head()

<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   No.                          160 non-null    int64  
 1   Kode Provinsi                160 non-null    int64  
 2   Provinsi                     160 non-null    str    
 3   Kode Kabupaten/Kota (BPS)    160 non-null    int64  
 4   Kabupaten/Kota               160 non-null    str    
 5   Kode Kecamatan (BPS)         160 non-null    int64  
 6   Kecamatan                    160 non-null    str    
 7   Kode Kecamatan (Kemendagri)  160 non-null    int64  
 8   Nama Kecamatan (Kemendagri)  160 non-null    str    
 9   Jumlah Rumah Tangga          160 non-null    float64
 10  Satuan                       160 non-null    str    
 11  Tahun                        160 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 15.1 KB
None


,No.,Kode Provinsi,Provinsi,Kode Kabupaten/Kota (BPS),Kabupaten/Kota,Kode Kecamatan (BPS),Kecamatan,Kode Kecamatan (Kemendagri),Nama Kecamatan (Kemendagri),Jumlah Rumah Tangga,Satuan,Tahun
0,1,32,JAWA BARAT,3201,KABUPATEN BOGOR,3201140,BABAKAN MADANG,320105,BABAKAN MADANG,18.887,Rumah Tangga,2020
1,2,32,JAWA BARAT,3201,KABUPATEN BOGOR,3201140,BABAKAN MADANG,320105,BABAKAN MADANG,91.636,Rumah Tangga,2021
2,3,32,JAWA BARAT,3201,KABUPATEN BOGOR,3201140,BABAKAN MADANG,320105,BABAKAN MADANG,91.348,Rumah Tangga,2022
3,4,32,JAWA BARAT,3201,KABUPATEN BOGOR,3201140,BABAKAN MADANG,320105,BABAKAN MADANG,47.348,Rumah Tangga,2023
4,5,32,JAWA BARAT,3201,KABUPATEN BOGOR,3201220,BOJONG GEDE,320113,BOJONG GEDE,25.752,Rumah Tangga,2020


Melihat tipe data/struktur data

In [7]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                       Non-Null Count  Dtype
---  ------                                                                                                                                                                                       --------------  -----
 0   No.;Kode Provinsi;Provinsi;Kode Kabupaten/Kota (BPS);Kabupaten/Kota;Kode Kecamatan (BPS);Kecamatan;Kode Kecamatan (Kemendagri);Nama Kecamatan (Kemendagri);Jumlah Rumah Tangga;Satuan;Tahun  160 non-null    str  
dtypes: str(1)
memory usage: 1.4 KB
None


Melihat contoh 5 baris pertama dari data tersebut

In [8]:
print(df.head())

  No.;Kode Provinsi;Provinsi;Kode Kabupaten/Kota (BPS);Kabupaten/Kota;Kode Kecamatan (BPS);Kecamatan;Kode Kecamatan (Kemendagri);Nama Kecamatan (Kemendagri);Jumlah Rumah Tangga;Satuan;Tahun
0  1;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201140;B...                                                                                                                                         
1  2;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201140;B...                                                                                                                                         
2  3;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201140;B...                                                                                                                                         
3  4;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201140;B...                                                                                                                                         
4  5;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201220;B.

Melihat statistik deskriptif kolom numerik

In [9]:
print(df.describe())

       No.;Kode Provinsi;Provinsi;Kode Kabupaten/Kota (BPS);Kabupaten/Kota;Kode Kecamatan (BPS);Kecamatan;Kode Kecamatan (Kemendagri);Nama Kecamatan (Kemendagri);Jumlah Rumah Tangga;Satuan;Tahun
count                                                 160                                                                                                                                         
unique                                                160                                                                                                                                         
top     1;32;JAWA BARAT;3201;KABUPATEN BOGOR;3201140;B...                                                                                                                                         
freq                                                    1                                                                                                                                         


Melihat data yang memiliki missing value

In [10]:
print(df.isnull().sum())

No.;Kode Provinsi;Provinsi;Kode Kabupaten/Kota (BPS);Kabupaten/Kota;Kode Kecamatan (BPS);Kecamatan;Kode Kecamatan (Kemendagri);Nama Kecamatan (Kemendagri);Jumlah Rumah Tangga;Satuan;Tahun    0
dtype: int64


Melihat baris data yang memiliki duplikasi (Identik penuk)

In [11]:
print(df.duplicated().sum())

0


Cek inkonsistensi penulisan, disini berdasarkan data yang sudah kita lihat berdasarkan `df.head()`

In [12]:
print("Cek missing values:\n", df_raw.isnull().sum())
print("\nJumlah kecamatan unik:", df_raw['Kecamatan'].nunique())
print("Daftar tahun:", sorted(df_raw['Tahun'].unique()))

Cek missing values:
 No.                            0
Kode Provinsi                  0
Provinsi                       0
Kode Kabupaten/Kota (BPS)      0
Kabupaten/Kota                 0
Kode Kecamatan (BPS)           0
Kecamatan                      0
Kode Kecamatan (Kemendagri)    0
Nama Kecamatan (Kemendagri)    0
Jumlah Rumah Tangga            0
Satuan                         0
Tahun                          0
dtype: int64

Jumlah kecamatan unik: 40
Daftar tahun: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


# 3. DATA PREPARATION
Membersihkan dan menyiapkan data agar siap dipakai untuk modeling. Termasuk: pembersihan data (cleaning), transformasi (scaling), dan pemilihan atribut (feature selection) yang akan dipakai model.

Pivot Data: Setiap baris menjadi 1 Kecamatan unik

In [15]:
df_pivot = df_raw.pivot(index='Kecamatan', columns='Tahun', values='Jumlah Rumah Tangga').reset_index()
df_pivot.columns = ['Kecamatan', 'RT_2020', 'RT_2021', 'RT_2022', 'RT_2023']

Rekayasa Fitur Indikator Ekonomi / Pasar

In [16]:
df_pivot['Ukuran_Pasar_2023'] = df_pivot['RT_2023']
df_pivot['Rata_Rata_Konsumen'] = df_pivot[['RT_2020', 'RT_2021', 'RT_2022', 'RT_2023']].mean(axis=1)
df_pivot['Pertumbuhan_Pasar_Pct'] = ((df_pivot['RT_2023'] - df_pivot['RT_2020']) / df_pivot['RT_2020']) * 100

Jumlah baris setelah menghapus duplikasi

In [17]:
print("Data setelah Pivot dan Fitur Baru:")
df_pivot.head()

Data setelah Pivot dan Fitur Baru:


,Kecamatan,RT_2020,RT_2021,RT_2022,RT_2023,Ukuran_Pasar_2023,Rata_Rata_Konsumen,Pertumbuhan_Pasar_Pct
0,BABAKAN MADANG,18.887,91.636,91.348,47.348,47.348,62.30475,150.690951
1,BOJONG GEDE,25.752,71.866,71.485,72.720,72.720,60.45575,182.385834
2,CARINGIN,58.309,68.239,68.037,92.553,92.553,71.78450,58.728498
3,CARIU,10.619,86.629,86.518,30.001,30.001,53.44175,182.521895
4,CIAMPEA,43.313,62.170,62.172,91.955,91.955,64.90250,112.303465


Memilih atribut yang dipakai untuk modeling. Karena tujuan bisnis adalah untuk melihat segmentasi pola usia, pendapatan dan perilaku belanja.

In [18]:
fitur_ekonomi = ['Ukuran_Pasar_2023', 'Rata_Rata_Konsumen', 'Pertumbuhan_Pasar_Pct']

Mencari nilai K-Means untuk skala tiap fitur agar tidak ada fitur yang (mendominasi) karena rentang nilai lebih besar seperti Pendapatan vs Usia

In [19]:
X = df_pivot[fitur_ekonomi].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [20]:
pd.DataFrame(X_scaled, columns=fitur_ekonomi).head()

,Ukuran_Pasar_2023,Rata_Rata_Konsumen,Pertumbuhan_Pasar_Pct
0,-0.825647,0.294575,0.128109
1,0.310876,0.154096,0.422972
2,1.199284,1.014807,-0.727434
3,-1.602695,-0.378799,0.424238
4,1.172497,0.491941,-0.229017


# 4. MODELING
Memilih teknik modeling, menentukan parameter (jumlah cluster), dan membangun model K-Means.

Membangun model dengan K=3

In [21]:
k_optimal = 3
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_pivot['Cluster'] = kmeans_final.fit_predict(X_scaled)

print("Distribusi Anggota Tiap Klaster:")
print(df_pivot['Cluster'].value_counts())

Distribusi Anggota Tiap Klaster:
Cluster
1    18
2    17
0     5
Name: count, dtype: int64


# 5. EVALUASI
Mengevaluasi kualitas model secara teknis (metrik clustering) dan mengevaluasi apakah hasilnya benar-benar menjawab tujuan bisnis

Evaluasi Silhoutte Score

In [22]:
skor_sil = silhouette_score(X_scaled, df_pivot['Cluster'])
print(f"Silhouette Score (k={k_optimal}): {skor_sil:.3f}")

profil_klaster = df_pivot.groupby('Cluster')[fitur_ekonomi].mean().round(2)
profil_klaster['Jumlah_Kecamatan'] = df_pivot['Cluster'].value_counts().sort_index()
print("\nProfil Rata-Rata per Klaster:")
print(profil_klaster)

Silhouette Score (k=3): 0.325

Profil Rata-Rata per Klaster:
         Ukuran_Pasar_2023  Rata_Rata_Konsumen  Pertumbuhan_Pasar_Pct  \
Cluster                                                                 
0                    98.48               60.18                 334.87   
1                    74.89               68.05                 111.23   
2                    46.51               47.72                 105.90   

         Jumlah_Kecamatan  
Cluster                    
0                       5  
1                      18  
2                      17  


# 6. Deployment
Menyajikan hasil dalam bentuk yang bisa dipakai oleh pengguna bisnis (misal tim marketing), termasuk rekomendasi tindakan per segmen, serta menyimpan data & model untuk dipakai ulang.

Label Segmen Wilayah

In [24]:
mapping_segmen = {
    0: 'Hub Komersial & Pertumbuhan Agresif (Tier 1)',
    1: 'Pusat Perdagangan Terbentuk / Skala Besar (Tier 2)',
    2: 'Pasar Perintis / Skala Terbatas (Tier 3)'
}
df_pivot['Nama_Segmen'] = df_pivot['Cluster'].map(mapping_segmen)

Simpan Data CSV & Model untuk Aplikasi Streamlit

In [26]:
import joblib

df_pivot.to_csv('hasil_clustering_kecamatan_bogor.csv', index=False)
joblib.dump(kmeans_final, 'model_kmeans_kecamatan.pkl')
joblib.dump(scaler, 'scaler_kecamatan.pkl')

print("Penyimpanan artefak model dan dataset segmentasi selesai!")

Penyimpanan artefak model dan dataset segmentasi selesai!
